# Arena Strategist — a vibe-coded, self-improving game-strategy agent

**5-Day AI Agents Intensive (Vibe Coding) — Capstone**

This notebook builds an **autonomous agent that discovers winning strategies for a
competitive 2-player simulation by optimising against a *calibrated league***. It is
the LLM-agent realisation of the eval-driven method that earned our team **silver
(top 5% of 4,689 teams)** in the *Orbit Wars* Kaggle simulation competition: a
*trustworthy evaluator in the inner loop, a memory of dead ends on the outside*.

Two LLM roles — a **Proposer** and a **Critic** (Gemini) — reason in natural language
about game strategy and emit structured "genomes". The agent scores each genome
through an **MCP-style tool surface**, remembers what worked and what didn't, and
self-corrects until it beats the strongest hand-built anchor.

### What it demonstrates (one per course day)
| Day | Concept | Where in this notebook |
|----|----|----|
| 1 | Vibe coding — natural language drives the build | Proposer/Critic system prompts in `agent/provider.py` |
| 2 | Tools & interoperability (MCP) | `agent/tools.py` — `list_tools()` / `call_tool()` |
| 3 | Context engineering: skills & memory | `agent/skills/*.md`, `agent/memory.py` (attempts + falsifications) |
| 4 | Agent quality & evaluation | `arena/league.py` — the calibrated Elo league IS the reward |
| 5 | Prototype → production | reproducible run, observable tool log, graceful degradation |

> **Runs with or without a key.** With a Gemini key (add a Kaggle Secret named
> `GOOGLE_API_KEY`) the Proposer/Critic are real LLM calls. Without one, a
> deterministic `Mock` strategist (guided hill-climb) runs instead, so this notebook
> executes top-to-bottom for the grader either way — the same fallback pattern the
> official course starter kernel uses.


## 1. Write out the project
Each module is written to disk from its own cell, so the code below is exactly what
runs — and exactly what was verified locally. (Pure Python; no heavy dependencies.)

In [ ]:
import os
for d in ('arena', 'agent', 'agent/skills'):
    os.makedirs(d, exist_ok=True)
print('dirs ready')

In [ ]:
%%writefile arena/__init__.py
"""Orbital Skirmish arena — a self-contained competitive 2-player simulation.

This package is the *testbed* for the Arena Strategist capstone agent. It is a
compact, fully deterministic re-creation (in spirit) of the kind of competitive
Kaggle simulation we tackled in Orbit Wars: cheap to simulate by the thousand,
with a genuine, calibrated skill ladder among scripted baseline agents.

Nothing here touches the live Orbit Wars competition or its locked submission —
this is an independent, MIT-spirit sandbox built for the course capstone.
"""

from .sim import Strategy, GameConfig, play_game, DEFAULT_GENOME, GENOME_KEYS
from .anchors import ANCHORS, get_anchor
from .league import League, LeagueResult

__all__ = [
    "Strategy",
    "GameConfig",
    "play_game",
    "DEFAULT_GENOME",
    "GENOME_KEYS",
    "ANCHORS",
    "get_anchor",
    "League",
    "LeagueResult",
]


In [ ]:
%%writefile arena/sim.py
"""Orbital Skirmish — a compact, fully deterministic 2-player strategy game.

Why this exists
---------------
The capstone agent needs a *competitive environment with a real skill ladder* so
that an LLM-proposed strategy can be scored objectively. We deliberately keep the
game:

* **deterministic** — given two strategies and a map, the outcome is fixed, so the
  league is reproducible (no Monte-Carlo noise to fight, the lesson we learned the
  hard way calibrating the Orbit Wars league);
* **cheap** — a full game is a few dozen integer turns, so we can run a 7-agent
  round-robin over a map pool in well under a second;
* **strategic** — economy / military / territory / defense form a
  rock-paper-scissors-ish loop, so no single pure strategy dominates and a genuine
  ladder emerges (miner < raider < expander < turtle < balanced < adaptive).

A *strategy* is an 8-number "genome" that parameterises a fixed, legible policy.
Representing strategies as a small vector (rather than arbitrary code) makes them
trivial to serialise, store in memory, mutate, and — crucially — to ask an LLM to
propose, while staying safe and sandbox-free.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Dict, List, Tuple

# --------------------------------------------------------------------------- #
# Strategy genome
# --------------------------------------------------------------------------- #

# The eight tunable knobs of the policy. Order matters: it defines the vector
# layout used by the league, the memory store, and the LLM proposal schema.
GENOME_KEYS: Tuple[str, ...] = (
    "mine",                 # base priority for MINE (+3 ore)
    "build",                # base priority for BUILD (-3 ore, +1 ship)
    "expand",               # base priority for EXPAND (-5 ore, +1 base = points)
    "fortify",              # base priority for FORTIFY (-2 ore, +1 shield)
    "raid",                 # per-ship priority for RAID (attack)
    "raid_threshold",       # min ships before RAID is considered
    "expand_late_bonus",    # EXPAND priority *= (1 + bonus * turn_fraction)
    "fortify_threat_scale", # FORTIFY priority *= (1 + scale * opponent_ships)
)

# A sensible, deliberately mediocre starting point the agent improves on.
DEFAULT_GENOME: Dict[str, float] = {
    "mine": 1.0,
    "build": 1.0,
    "expand": 1.0,
    "fortify": 1.0,
    "raid": 1.0,
    "raid_threshold": 3,
    "expand_late_bonus": 0.0,
    "fortify_threat_scale": 0.0,
}


@dataclass
class Strategy:
    """A named policy. `genome` holds the eight knobs in GENOME_KEYS."""

    name: str
    genome: Dict[str, float] = field(default_factory=lambda: dict(DEFAULT_GENOME))
    note: str = ""  # free-text rationale (the LLM fills this in)

    def normalised(self) -> Dict[str, float]:
        """Return a genome with every key present and clamped to safe ranges."""
        g = dict(DEFAULT_GENOME)
        for k, v in (self.genome or {}).items():
            if k in DEFAULT_GENOME:
                g[k] = v
        # Clamp to keep the policy well-defined and the search bounded.
        for k in ("mine", "build", "expand", "fortify", "raid"):
            g[k] = float(max(0.0, min(10.0, g[k])))
        g["raid_threshold"] = int(max(1, min(20, round(g["raid_threshold"]))))
        g["expand_late_bonus"] = float(max(0.0, min(5.0, g["expand_late_bonus"])))
        g["fortify_threat_scale"] = float(max(0.0, min(2.0, g["fortify_threat_scale"])))
        return g

    def vector(self) -> List[float]:
        g = self.normalised()
        return [float(g[k]) for k in GENOME_KEYS]


# --------------------------------------------------------------------------- #
# Game configuration ("map")
# --------------------------------------------------------------------------- #

@dataclass(frozen=True)
class GameConfig:
    """A single 'map'. Varying these gives us a map pool, like Orbit Wars."""

    turns: int = 50
    start_ore: int = 10
    mine_yield: int = 3
    build_cost: int = 3
    expand_cost: int = 5
    fortify_cost: int = 2


# --------------------------------------------------------------------------- #
# Engine
# --------------------------------------------------------------------------- #

# Action codes
MINE, BUILD, EXPAND, FORTIFY, RAID = 0, 1, 2, 3, 4
ACTION_NAMES = {MINE: "MINE", BUILD: "BUILD", EXPAND: "EXPAND",
                FORTIFY: "FORTIFY", RAID: "RAID"}


class _Side:
    __slots__ = ("ore", "ships", "bases", "shield")

    def __init__(self, start_ore: int):
        self.ore = start_ore
        self.ships = 0
        self.bases = 0
        self.shield = 0


def _choose_action(me: _Side, opp: _Side, g: Dict[str, float],
                   turn: int, cfg: GameConfig) -> int:
    """Score every legal action by the genome and return the best one.

    MINE is always legal (costs nothing), so the policy is always defined.
    Ties break by a fixed action order, keeping the game deterministic.
    """
    turn_frac = turn / max(1, cfg.turns)
    scores: List[Tuple[float, int]] = []

    # MINE — always available.
    scores.append((g["mine"], MINE))

    # BUILD — needs ore.
    if me.ore >= cfg.build_cost:
        scores.append((g["build"], BUILD))

    # EXPAND — needs ore; weighted up in the late game if the genome says so.
    if me.ore >= cfg.expand_cost:
        s = g["expand"] * (1.0 + g["expand_late_bonus"] * turn_frac)
        scores.append((s, EXPAND))

    # FORTIFY — needs ore; weighted up when the opponent has a fleet.
    if me.ore >= cfg.fortify_cost:
        s = g["fortify"] * (1.0 + g["fortify_threat_scale"] * opp.ships)
        scores.append((s, FORTIFY))

    # RAID — needs a fleet at or above the threshold.
    if me.ships >= g["raid_threshold"] and me.ships >= 1:
        s = g["raid"] * me.ships
        scores.append((s, RAID))

    # argmax with stable tiebreak (lowest action code wins ties).
    best_score, best_action = scores[0]
    for s, a in scores[1:]:
        if s > best_score + 1e-12 or (abs(s - best_score) <= 1e-12 and a < best_action):
            best_score, best_action = s, a
    return best_action


def _apply_economy(side: _Side, action: int, cfg: GameConfig) -> None:
    if action == MINE:
        side.ore += cfg.mine_yield
    elif action == BUILD:
        side.ore -= cfg.build_cost
        side.ships += 1
    elif action == EXPAND:
        side.ore -= cfg.expand_cost
        side.bases += 1
    elif action == FORTIFY:
        side.ore -= cfg.fortify_cost
        side.shield += 1
    # RAID is resolved in the combat phase.


def _resolve_raid(attacker: _Side, defender: _Side) -> None:
    """Apply one raid. Uses post-economy snapshots taken by the caller.

    Net attacking ships (fleet minus the defender's standing shields) destroy one
    base each. Fleets and shields are persistent — the cost of raiding is the turn
    you didn't spend on economy, which keeps a credible offence in tension with
    a credible defence.
    """
    damage = max(0, attacker.ships - defender.shield)
    defender.bases = max(0, defender.bases - damage)


def play_game(a: Strategy, b: Strategy, cfg: GameConfig) -> Tuple[int, dict]:
    """Play one game. Returns (result, detail).

    result is +1 if `a` wins, -1 if `b` wins, 0 for a draw (from a's POV).
    """
    ga, gb = a.normalised(), b.normalised()
    A, B = _Side(cfg.start_ore), _Side(cfg.start_ore)

    for turn in range(cfg.turns):
        # Simultaneous decisions from the same snapshot.
        act_a = _choose_action(A, B, ga, turn, cfg)
        act_b = _choose_action(B, A, gb, turn, cfg)

        # Economy resolves first (so a same-turn FORTIFY can defend this turn).
        _apply_economy(A, act_a, cfg)
        _apply_economy(B, act_b, cfg)

        # Combat resolves from a frozen post-economy snapshot so both raids use
        # the same fleet/shield sizes (truly simultaneous). Net attacking ships
        # (fleet minus standing shields) destroy one base each; fleets and shields
        # persist — the cost of a raid is the economy turn you forfeited.
        a_ships, b_ships = A.ships, B.ships
        a_shield, b_shield = A.shield, B.shield
        if act_a == RAID:
            B.bases = max(0, B.bases - max(0, a_ships - b_shield))
        if act_b == RAID:
            A.bases = max(0, A.bases - max(0, b_ships - a_shield))

    # Scoring: territory first, salvage value as tiebreak.
    detail = {
        "a": {"ore": A.ore, "ships": A.ships, "bases": A.bases, "shield": A.shield},
        "b": {"ore": B.ore, "ships": B.ships, "bases": B.bases, "shield": B.shield},
    }
    if A.bases != B.bases:
        return (1 if A.bases > B.bases else -1), detail
    salv_a = A.ore + A.ships * 3 + A.shield
    salv_b = B.ore + B.ships * 3 + B.shield
    if salv_a != salv_b:
        return (1 if salv_a > salv_b else -1), detail
    return 0, detail


def play_match(a: Strategy, b: Strategy, pool: List[GameConfig]) -> Tuple[int, int, int]:
    """Play `a` vs `b` across a map pool, both sides of each map (to cancel any
    first-mover asymmetry). Returns (wins_a, draws, wins_b)."""
    wa = d = wb = 0
    for cfg in pool:
        for x, y, flip in ((a, b, 1), (b, a, -1)):
            r, _ = play_game(x, y, cfg)
            r *= flip  # re-orient to a's POV
            if r > 0:
                wa += 1
            elif r < 0:
                wb += 1
            else:
                d += 1
    return wa, d, wb


In [ ]:
%%writefile arena/anchors.py
"""The calibrated anchor ladder.

These hand-written strategies span the skill spectrum from naive to strong. Their
*order* on the league is what makes the league trustworthy: if a brand-new eval
harness can't reproduce a sensible anchor ordering, you can't trust what it says
about a novel strategy. This is the single most important lesson we carried over
from Orbit Wars — calibrate the league against known anchors *first*, then let it
judge new agents.

The agent under test must climb this ladder. `adaptive` is the top anchor (our
local stand-in for the Orbit Wars "producer" ceiling) — beating it is the goal.
"""

from __future__ import annotations

from typing import Dict, List

from .sim import Strategy

# Each anchor is a genome (see arena.sim.GENOME_KEYS) + a short rationale.
_ANCHOR_SPECS: List[Dict] = [
    {
        "name": "naive_miner",
        "note": "Hoards ore, never converts it to points. Bottom of the ladder.",
        "genome": {"mine": 5.0, "build": 0.2, "expand": 0.2, "fortify": 0.1,
                   "raid": 0.1, "raid_threshold": 20},
    },
    {
        "name": "all_in_raider",
        "note": "Builds a fleet and raids relentlessly; no economy backbone.",
        "genome": {"mine": 1.2, "build": 3.0, "expand": 0.3, "fortify": 0.2,
                   "raid": 2.5, "raid_threshold": 2},
    },
    {
        "name": "greedy_expander",
        "note": "Expands at every chance, undefended. Scores fast, dies to raids.",
        "genome": {"mine": 1.5, "build": 0.3, "expand": 3.0, "fortify": 0.3,
                   "raid": 0.2, "raid_threshold": 20, "expand_late_bonus": 0.5},
    },
    {
        "name": "turtle",
        "note": "Expands behind shields; reactive fortification vs fleets.",
        "genome": {"mine": 1.6, "build": 0.6, "expand": 2.0, "fortify": 1.8,
                   "raid": 0.3, "raid_threshold": 8, "fortify_threat_scale": 0.6},
    },
    {
        "name": "balanced",
        "note": "Mixed economy into expansion, with a credible raiding deterrent.",
        "genome": {"mine": 1.8, "build": 1.4, "expand": 2.0, "fortify": 1.0,
                   "raid": 1.2, "raid_threshold": 4, "expand_late_bonus": 0.4,
                   "fortify_threat_scale": 0.4},
    },
    {
        "name": "adaptive",
        "note": "Top anchor: economy first, expands late, fortifies under threat, "
                "raids only with a decisive fleet. The local 'ceiling' to beat.",
        "genome": {"mine": 2.2, "build": 1.5, "expand": 1.9, "fortify": 1.2,
                   "raid": 1.6, "raid_threshold": 5, "expand_late_bonus": 1.2,
                   "fortify_threat_scale": 0.8},
    },
]

ANCHORS: List[Strategy] = [
    Strategy(name=s["name"], genome=dict(s["genome"]), note=s["note"])
    for s in _ANCHOR_SPECS
]

_BY_NAME = {s.name: s for s in ANCHORS}


def get_anchor(name: str) -> Strategy:
    return _BY_NAME[name]


In [ ]:
%%writefile arena/league.py
"""The calibrated league — the agent's reward signal.

Given a set of strategies and a map pool, we run a full round-robin (each pair on
every map, both sides) and convert the win matrix into Elo-style ratings with a
deterministic Bradley-Terry fit. No randomness, so the ratings are perfectly
reproducible — exactly what you want when an LLM is going to optimise against them.

`evaluate_candidate` is the function the agent's tools call: drop a new strategy in
with the anchors, return its rating and its win-rate versus the top anchor.
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

from .anchors import ANCHORS
from .sim import GameConfig, Strategy, play_match

# A small, fixed map pool. Varying turns / starting ore / mine yield stresses
# different economic tempos so a strategy can't overfit a single map.
DEFAULT_POOL: List[GameConfig] = [
    GameConfig(turns=40, start_ore=8, mine_yield=3),
    GameConfig(turns=50, start_ore=10, mine_yield=3),
    GameConfig(turns=60, start_ore=10, mine_yield=2),
    GameConfig(turns=50, start_ore=14, mine_yield=3),
    GameConfig(turns=70, start_ore=6, mine_yield=4),
]


@dataclass
class LeagueResult:
    names: List[str]
    elo: Dict[str, float]
    wins: Dict[str, int]
    games: Dict[str, int]
    winrate_matrix: Dict[str, Dict[str, float]] = field(default_factory=dict)

    def table(self) -> str:
        rows = sorted(self.names, key=lambda n: -self.elo[n])
        out = ["  rank  strategy            elo     W     played  winrate",
               "  ----  ------------------  ------  ----  ------  -------"]
        for i, n in enumerate(rows, 1):
            wr = self.wins[n] / self.games[n] if self.games[n] else 0.0
            out.append(f"  {i:>4}  {n:<18}  {self.elo[n]:>6.0f}  "
                       f"{self.wins[n]:>4}  {self.games[n]:>6}  {wr:>6.1%}")
        return "\n".join(out)


def _bradley_terry_elo(names: List[str],
                       score: Dict[str, Dict[str, float]],
                       played: Dict[str, Dict[str, int]],
                       iters: int = 500) -> Dict[str, float]:
    """Fit Bradley-Terry strengths from a win matrix, return Elo-scaled ratings.

    `score[i][j]` = points i took off j (win=1, draw=0.5). Deterministic MM update.
    """
    strength = {n: 1.0 for n in names}
    for _ in range(iters):
        new = {}
        for i in names:
            wins_i = sum(score[i][j] for j in names if j != i)
            denom = 0.0
            for j in names:
                if j == i:
                    continue
                games_ij = played[i][j]
                if games_ij:
                    denom += games_ij / (strength[i] + strength[j])
            new[i] = wins_i / denom if denom > 0 else strength[i]
        # Normalise geometric mean to 1 to keep the fit from drifting.
        gm = math.exp(sum(math.log(max(v, 1e-9)) for v in new.values()) / len(new))
        strength = {n: new[n] / gm for n in names}
    # Map log-strength to Elo (400/ln10 per natural log unit), centred at 1000.
    scale = 400.0 / math.log(10.0)
    return {n: 1000.0 + scale * math.log(max(strength[n], 1e-9)) for n in names}


class League:
    """Run round-robins and evaluate candidates against the anchor ladder."""

    def __init__(self, anchors: List[Strategy] | None = None,
                 pool: List[GameConfig] | None = None):
        self.anchors = anchors if anchors is not None else list(ANCHORS)
        self.pool = pool if pool is not None else list(DEFAULT_POOL)

    def run(self, extra: List[Strategy] | None = None) -> LeagueResult:
        strategies = list(self.anchors) + list(extra or [])
        names = [s.name for s in strategies]
        score = {i: {j: 0.0 for j in names} for i in names}
        played = {i: {j: 0 for j in names} for i in names}
        wins = {n: 0 for n in names}
        games = {n: 0 for n in names}
        wr = {i: {} for i in names}

        for a in range(len(strategies)):
            for b in range(a + 1, len(strategies)):
                sa, sb = strategies[a], strategies[b]
                wa, dr, wb = play_match(sa, sb, self.pool)
                n_games = wa + dr + wb
                # accumulate symmetric stats
                score[sa.name][sb.name] += wa + 0.5 * dr
                score[sb.name][sa.name] += wb + 0.5 * dr
                played[sa.name][sb.name] += n_games
                played[sb.name][sa.name] += n_games
                wins[sa.name] += wa
                wins[sb.name] += wb
                games[sa.name] += n_games
                games[sb.name] += n_games
                wr[sa.name][sb.name] = wa / n_games if n_games else 0.0
                wr[sb.name][sa.name] = wb / n_games if n_games else 0.0

        elo = _bradley_terry_elo(names, score, played)
        return LeagueResult(names=names, elo=elo, wins=wins, games=games,
                            winrate_matrix=wr)

    def top_anchor(self) -> Strategy:
        """Highest-rated anchor in an anchors-only league (the ceiling to beat)."""
        res = self.run()
        best = max(self.anchors, key=lambda s: res.elo[s.name])
        return best

    def evaluate_candidate(self, candidate: Strategy) -> Dict:
        """Score one candidate against the anchor ladder. This is the core signal
        the agent optimises. Returns elo, rank, and win-rate vs the top anchor."""
        res = self.run(extra=[candidate])
        ranked = sorted(res.names, key=lambda n: -res.elo[n])
        rank = ranked.index(candidate.name) + 1
        top = max(self.anchors, key=lambda s: res.elo[s.name]).name
        return {
            "name": candidate.name,
            "elo": round(res.elo[candidate.name], 1),
            "rank": rank,
            "field_size": len(res.names),
            "top_anchor": top,
            "top_anchor_elo": round(res.elo[top], 1),
            "winrate_vs_top": round(res.winrate_matrix[candidate.name].get(top, 0.0), 3),
            "beats_top_anchor": res.elo[candidate.name] > res.elo[top],
            "winrate_overall": round(res.wins[candidate.name] / res.games[candidate.name], 3)
            if res.games[candidate.name] else 0.0,
        }


if __name__ == "__main__":
    # Calibration check: anchors-only league should produce a sensible ladder.
    league = League()
    result = league.run()
    print("Anchor-only calibrated league:\n")
    print(result.table())
    print(f"\nTop anchor (ceiling to beat): {league.top_anchor().name}")


In [ ]:
%%writefile agent/__init__.py
"""Arena Strategist — the capstone agent.

A vibe-coded, tool-using, multi-agent system that discovers winning strategies
for a competitive simulation by optimising against a *calibrated league* — the
same eval-driven methodology that took us to silver (top 5%) in the Orbit Wars
Kaggle competition.

Layers (mapping to the 5 course days):
  * provider   — Gemini-backed Proposer/Critic, with a no-API-key Mock fallback   (D1 vibe coding)
  * tools      — an MCP-style tool server wrapping the arena + league             (D2 tools/MCP)
  * memory     — persistent strategy store + falsification log                    (D3 skills/memory)
  * strategist — orchestration loop with self-correction; the league is the eval  (D4 quality/eval)
  * run_demo   — reproducible, observable end-to-end run                          (D5 production)
"""

from .provider import get_provider, StrategyProvider, MockProvider, GeminiProvider
from .memory import StrategyMemory
from .tools import ArenaToolServer
from .strategist import ArenaStrategist, RunReport

__all__ = [
    "get_provider",
    "StrategyProvider",
    "MockProvider",
    "GeminiProvider",
    "StrategyMemory",
    "ArenaToolServer",
    "ArenaStrategist",
    "RunReport",
]


In [ ]:
%%writefile agent/provider.py
"""LLM provider abstraction: a Gemini-backed strategist with a Mock fallback.

The whole point of *vibe coding* is that natural language drives the work — here,
two LLM "roles" (a Proposer and a Critic) reason about game strategy in words and
emit structured genomes. But a capstone notebook has to *run for the grader even
without secrets*, so we ship a deterministic `MockProvider` that performs guided
hill-climbing. The official course starter kernel uses exactly this pattern (a
`MockLLMContext`) so the notebook executes top-to-bottom with or without a key.

Set a key (`GOOGLE_API_KEY` / `GEMINI_API_KEY`, or a Kaggle Secret of the same
name) and `get_provider()` returns the real `GeminiProvider`; otherwise you get
the `MockProvider` and an explanatory log line.
"""

from __future__ import annotations

import json
import os
import random
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from arena.sim import DEFAULT_GENOME, GENOME_KEYS, Strategy


# --------------------------------------------------------------------------- #
# Shared context + result types
# --------------------------------------------------------------------------- #

@dataclass
class ProposalContext:
    """Everything a role needs to reason about the next move."""

    top_anchor: str
    top_anchor_elo: float
    anchor_summary: List[Dict]               # [{name, elo, note}]
    skill_text: str                          # the strategy-heuristics skill file
    best_genome: Optional[Dict] = None       # best genome found so far
    best_elo: Optional[float] = None
    history: List[Dict] = field(default_factory=list)   # recent {genome, elo, beats_top}
    falsifications: List[str] = field(default_factory=list)  # what didn't work
    round_index: int = 0
    budget: int = 1


@dataclass
class Proposal:
    genome: Dict[str, float]
    rationale: str
    source: str  # "gemini" | "mock"


@dataclass
class Critique:
    verdict: str            # "keep" | "mutate" | "discard"
    reasoning: str
    suggested_changes: Dict[str, float] = field(default_factory=dict)


def _coerce_genome(raw: Dict) -> Dict[str, float]:
    """Make any dict into a valid genome (fill missing keys, drop unknowns)."""
    g = dict(DEFAULT_GENOME)
    for k in GENOME_KEYS:
        if k in raw:
            try:
                g[k] = float(raw[k])
            except (TypeError, ValueError):
                pass
    g["raid_threshold"] = int(round(g["raid_threshold"]))
    # Round the continuous knobs for clean storage/display (ample resolution).
    for k in GENOME_KEYS:
        if k != "raid_threshold":
            g[k] = round(g[k], 3)
    return g


# --------------------------------------------------------------------------- #
# Base class
# --------------------------------------------------------------------------- #

class StrategyProvider:
    name = "base"

    def propose(self, ctx: ProposalContext) -> Proposal:
        raise NotImplementedError

    def critique(self, candidate: Strategy, eval_result: Dict,
                 ctx: ProposalContext) -> Critique:
        raise NotImplementedError


# --------------------------------------------------------------------------- #
# Mock provider — deterministic guided hill-climber (no API key required)
# --------------------------------------------------------------------------- #

class MockProvider(StrategyProvider):
    """A real optimiser standing in for the LLM. Hill-climbs the genome around the
    best-known strategy, with periodic exploratory restarts to escape local
    optima. Deterministic given the round index, so runs are reproducible."""

    name = "mock"

    # A sensible economy-leaning seed so we start above mediocrity.
    _SEED = {"mine": 1.8, "build": 1.6, "expand": 2.2, "fortify": 1.3,
             "raid": 1.6, "raid_threshold": 3, "expand_late_bonus": 1.0,
             "fortify_threat_scale": 0.6}

    _STEP = {"mine": 0.5, "build": 0.5, "expand": 0.5, "fortify": 0.5,
             "raid": 0.5, "raid_threshold": 1, "expand_late_bonus": 0.4,
             "fortify_threat_scale": 0.3}

    def propose(self, ctx: ProposalContext) -> Proposal:
        rng = random.Random(1000 + ctx.round_index)
        base = dict(ctx.best_genome) if ctx.best_genome else dict(self._SEED)

        if ctx.best_genome is None:
            return Proposal(genome=_coerce_genome(base),
                            rationale="Seed: economy-first with late expansion and "
                                      "a reactive defence — a balanced opening to climb from.",
                            source=self.name)

        # Every 5th round, take a bigger exploratory jump (anneal the temperature).
        explore = (ctx.round_index % 5 == 0)
        temp = 1.6 if explore else 0.6 * (1.0 - 0.5 * ctx.round_index / max(1, ctx.budget))
        knobs = rng.sample(list(GENOME_KEYS), k=3 if explore else 2)

        g = dict(base)
        changed = {}
        for k in knobs:
            delta = rng.choice([-1.0, 1.0]) * self._STEP[k] * temp * rng.uniform(0.6, 1.5)
            if k == "raid_threshold":
                delta = rng.choice([-1, 1])
            g[k] = g[k] + delta
            changed[k] = round(g[k] - base[k], 2)

        verb = "Exploratory jump" if explore else "Hill-climb"
        nudges = ", ".join(f"{k} {'+' if v >= 0 else ''}{v}" for k, v in changed.items())
        rationale = (f"{verb} from best (elo {ctx.best_elo:.0f}): nudging {nudges} "
                     f"to push win-rate vs {ctx.top_anchor}.")
        return Proposal(genome=_coerce_genome(g), rationale=rationale, source=self.name)

    def critique(self, candidate: Strategy, eval_result: Dict,
                 ctx: ProposalContext) -> Critique:
        improved = (ctx.best_elo is None) or (eval_result["elo"] > ctx.best_elo)
        if eval_result["beats_top_anchor"] and improved:
            return Critique("keep",
                            f"New best: elo {eval_result['elo']} beats {ctx.top_anchor} "
                            f"(win-rate {eval_result['winrate_vs_top']:.0%}). Adopt as base.")
        if improved:
            return Critique("keep",
                            f"Improves elo to {eval_result['elo']} but doesn't yet clear "
                            f"{ctx.top_anchor}. Keep as the new climbing base.")
        # Clearly weak (low absolute win-rate, or far below our best) — discard and
        # let the orchestrator record a falsification so we never re-explore it.
        far_below = ctx.best_elo is not None and eval_result["elo"] < ctx.best_elo - 200
        if eval_result["winrate_overall"] < 0.30 or far_below:
            return Critique("discard",
                            f"elo {eval_result['elo']} ({eval_result['winrate_overall']:.0%} "
                            f"overall win-rate); well below the best line, a dead end.")
        # Otherwise mutate: diagnose a direction from the eval signal.
        changes: Dict[str, float] = {}
        if eval_result["winrate_vs_top"] < 0.5:
            changes["fortify"] = +0.5      # losing to the ceiling -> shore up defence
            changes["raid"] = +0.5         # ...and add bite
        return Critique("mutate",
                        f"elo {eval_result['elo']} below best {ctx.best_elo:.0f}; "
                        f"mutate from the current best instead.",
                        suggested_changes=changes)


# --------------------------------------------------------------------------- #
# Gemini provider — real LLM-driven Proposer / Critic
# --------------------------------------------------------------------------- #

_PROPOSER_SYSTEM = """You are the Proposer in a two-agent strategy-discovery loop \
for a competitive 2-player game called Orbital Skirmish.

Each turn a player picks ONE action by the highest weighted score:
- MINE (+3 ore), BUILD (-3 ore, +1 ship), EXPAND (-5 ore, +1 base = victory points),
  FORTIFY (-2 ore, +1 standing shield), RAID (net ships above the enemy's shield
  each destroy one of their bases). Fleets and shields persist.

A STRATEGY is an 8-number genome controlling that policy:
  mine, build, expand, fortify, raid (priorities >=0),
  raid_threshold (min ships before raiding, integer 1-20),
  expand_late_bonus (expand priority grows toward the endgame, 0-5),
  fortify_threat_scale (fortify priority grows with enemy fleet size, 0-2).

Your job: propose the next genome to TEST so it climbs an Elo league of anchor
strategies and beats the top anchor. Reason briefly, then OUTPUT STRICT JSON:
{"genome": {"mine":..,"build":..,"expand":..,"fortify":..,"raid":..,
"raid_threshold":..,"expand_late_bonus":..,"fortify_threat_scale":..},
"rationale":"one sentence"}"""

_CRITIC_SYSTEM = """You are the Critic in a two-agent strategy-discovery loop. \
Given a candidate genome and its league evaluation, decide whether to keep it as \
the new climbing base, mutate from the current best, or discard it. Consider its \
Elo, whether it beats the top anchor, and its win-rate versus the top anchor.
OUTPUT STRICT JSON: {"verdict":"keep|mutate|discard","reasoning":"one sentence",
"suggested_changes":{"<knob>": <signed delta>, ...}}"""


class GeminiProvider(StrategyProvider):
    """Calls Gemini for the Proposer and Critic roles. Falls back to a coerced
    genome if the model returns unparseable output, so the loop never stalls."""

    name = "gemini"

    def __init__(self, api_key: str, model: Optional[str] = None):
        import google.generativeai as genai  # lazy: only needed on this path
        self._genai = genai
        genai.configure(api_key=api_key)
        self.model_name = model or os.environ.get("GEMINI_MODEL", "gemini-2.0-flash")
        self._proposer = genai.GenerativeModel(self.model_name,
                                               system_instruction=_PROPOSER_SYSTEM)
        self._critic = genai.GenerativeModel(self.model_name,
                                             system_instruction=_CRITIC_SYSTEM)
        self._mock = MockProvider()  # safety net for parse failures

    def _json_call(self, model, prompt: str) -> Optional[Dict]:
        try:
            resp = model.generate_content(
                prompt,
                generation_config={"response_mime_type": "application/json",
                                   "temperature": 0.9, "max_output_tokens": 512},
            )
            return json.loads(resp.text)
        except Exception as exc:  # parse error, rate limit, etc. — degrade gracefully
            print(f"    [gemini] call failed ({type(exc).__name__}); using fallback")
            return None

    def _render_context(self, ctx: ProposalContext) -> str:
        anchors = "\n".join(f"  - {a['name']}: elo {a['elo']:.0f} — {a['note']}"
                            for a in ctx.anchor_summary)
        hist = "\n".join(
            f"  - elo {h['elo']:.0f} {'(beat ceiling)' if h['beats_top'] else ''}: "
            f"{json.dumps({k: round(v, 2) for k, v in h['genome'].items()})}"
            for h in ctx.history[-6:]) or "  (none yet)"
        false = "\n".join(f"  - {f}" for f in ctx.falsifications[-6:]) or "  (none yet)"
        best = (json.dumps({k: round(v, 2) for k, v in ctx.best_genome.items()})
                if ctx.best_genome else "none")
        return (f"SKILL — strategy heuristics:\n{ctx.skill_text}\n\n"
                f"ANCHOR LADDER (beat the top one, '{ctx.top_anchor}' at "
                f"elo {ctx.top_anchor_elo:.0f}):\n{anchors}\n\n"
                f"BEST SO FAR: elo {ctx.best_elo} genome {best}\n\n"
                f"RECENT ATTEMPTS:\n{hist}\n\n"
                f"FALSIFIED (don't repeat):\n{false}\n\n"
                f"Round {ctx.round_index + 1} of {ctx.budget}.")

    def propose(self, ctx: ProposalContext) -> Proposal:
        prompt = self._render_context(ctx) + "\n\nPropose the next genome to test."
        data = self._json_call(self._proposer, prompt)
        if not data or "genome" not in data:
            return self._mock.propose(ctx)
        return Proposal(genome=_coerce_genome(data["genome"]),
                        rationale=str(data.get("rationale", ""))[:240],
                        source=self.name)

    def critique(self, candidate: Strategy, eval_result: Dict,
                 ctx: ProposalContext) -> Critique:
        prompt = (self._render_context(ctx) +
                  f"\n\nCANDIDATE genome: "
                  f"{json.dumps({k: round(v, 2) for k, v in candidate.normalised().items()})}"
                  f"\nEVALUATION: {json.dumps(eval_result)}\n\nCritique it.")
        data = self._json_call(self._critic, prompt)
        if not data or "verdict" not in data:
            return self._mock.critique(candidate, eval_result, ctx)
        verdict = str(data.get("verdict", "mutate")).lower()
        if verdict not in ("keep", "mutate", "discard"):
            verdict = "mutate"
        changes = {}
        for k, v in (data.get("suggested_changes") or {}).items():
            if k in GENOME_KEYS:
                try:
                    changes[k] = float(v)
                except (TypeError, ValueError):
                    pass
        return Critique(verdict, str(data.get("reasoning", ""))[:240], changes)


# --------------------------------------------------------------------------- #
# Factory
# --------------------------------------------------------------------------- #

def _find_api_key() -> Optional[str]:
    for var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
        if os.environ.get(var):
            return os.environ[var]
    # Kaggle Secrets, if running in a Kaggle notebook.
    try:
        from kaggle_secrets import UserSecretsClient  # type: ignore
        client = UserSecretsClient()
        for var in ("GOOGLE_API_KEY", "GEMINI_API_KEY"):
            try:
                val = client.get_secret(var)
                if val:
                    return val
            except Exception:
                pass
    except Exception:
        pass
    return None


def get_provider(force_mock: bool = False) -> StrategyProvider:
    """Return a GeminiProvider if a key + SDK are available, else a MockProvider."""
    if force_mock:
        print("[provider] forced Mock strategist (deterministic, no API).")
        return MockProvider()
    key = _find_api_key()
    if not key:
        print("[provider] no GOOGLE_API_KEY/GEMINI_API_KEY found → Mock strategist "
              "(deterministic hill-climber). Add a key to enable Gemini.")
        return MockProvider()
    try:
        import google.generativeai  # noqa: F401
    except Exception:
        print("[provider] google-generativeai not installed → Mock strategist. "
              "`pip install google-generativeai` to enable Gemini.")
        return MockProvider()
    prov = GeminiProvider(api_key=key)
    print(f"[provider] Gemini strategist active (model: {prov.model_name}).")
    return prov


In [ ]:
%%writefile agent/memory.py
"""Persistent strategy memory + falsification log (course Day 3: memory & state).

Two things the agent remembers across turns (and across runs, via JSON on disk):

1. **Attempts** — every genome tried, its league evaluation, who proposed it and
   the critic's verdict. This is the agent's episodic memory; the Proposer reads
   the recent slice to avoid re-treading ground and to hill-climb from the best.

2. **Falsifications** — short, human-readable lessons ("raid-only with no economy
   tops out ~elo 880"). This is the durable, distilled memory. It's the single
   highest-leverage idea we took from Orbit Wars: *write down what DOESN'T work*
   so neither a human nor an LLM wastes budget re-exploring dead ends.
"""

from __future__ import annotations

import json
import os
from dataclasses import asdict, dataclass, field
from typing import Dict, List, Optional

from arena.sim import Strategy


@dataclass
class Attempt:
    round_index: int
    name: str
    genome: Dict[str, float]
    elo: float
    rank: int
    beats_top_anchor: bool
    winrate_vs_top: float
    rationale: str
    source: str          # "gemini" | "mock"
    verdict: str         # critic verdict


class StrategyMemory:
    def __init__(self, path: Optional[str] = None):
        self.path = path
        self.attempts: List[Attempt] = []
        self.falsifications: List[str] = []
        self.best: Optional[Attempt] = None
        if path and os.path.exists(path):
            self.load()

    # -- recording ---------------------------------------------------------- #
    def record(self, attempt: Attempt) -> None:
        self.attempts.append(attempt)
        if (self.best is None) or (attempt.elo > self.best.elo):
            self.best = attempt
        self.save()

    def add_falsification(self, lesson: str) -> None:
        if lesson and lesson not in self.falsifications:
            self.falsifications.append(lesson)
            self.save()

    # -- recall (used to build the Proposer/Critic context) ----------------- #
    def recent(self, n: int = 6) -> List[Dict]:
        return [{"genome": a.genome, "elo": a.elo, "beats_top": a.beats_top_anchor}
                for a in self.attempts[-n:]]

    def best_genome(self) -> Optional[Dict]:
        return dict(self.best.genome) if self.best else None

    def best_elo(self) -> Optional[float]:
        return self.best.elo if self.best else None

    def already_tried(self, genome: Dict[str, float], tol: float = 1e-6) -> bool:
        for a in self.attempts:
            if all(abs(a.genome.get(k, 0) - genome.get(k, 0)) <= tol for k in genome):
                return True
        return False

    # -- persistence -------------------------------------------------------- #
    def save(self) -> None:
        if not self.path:
            return
        data = {
            "attempts": [asdict(a) for a in self.attempts],
            "falsifications": self.falsifications,
            "best": asdict(self.best) if self.best else None,
        }
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)

    def load(self) -> None:
        with open(self.path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.attempts = [Attempt(**a) for a in data.get("attempts", [])]
        self.falsifications = data.get("falsifications", [])
        self.best = Attempt(**data["best"]) if data.get("best") else None

    def summary(self) -> str:
        if not self.attempts:
            return "memory: empty"
        return (f"memory: {len(self.attempts)} attempts, "
                f"best elo {self.best.elo:.0f} ({self.best.name}), "
                f"{len(self.falsifications)} falsifications recorded")


In [ ]:
%%writefile agent/tools.py
"""An MCP-style tool server wrapping the arena (course Day 2: tools & MCP).

The agent never touches the simulator directly — it acts only through this tool
surface, exactly the discipline the Model Context Protocol encourages: a small set
of well-described, schema-typed tools, a `list_tools()` (MCP `tools/list`) and a
`call_tool()` (MCP `tools/call`) dispatcher, and a log of every call for
observability. Swapping this class for a real `mcp` stdio server is mechanical —
the schemas are already in MCP shape.
"""

from __future__ import annotations

from typing import Dict, List

from arena.league import League
from arena.sim import GENOME_KEYS, Strategy

_GENOME_PROPERTIES = {
    "mine": {"type": "number", "description": "priority for MINE (+3 ore)"},
    "build": {"type": "number", "description": "priority for BUILD (-3 ore, +1 ship)"},
    "expand": {"type": "number", "description": "priority for EXPAND (-5 ore, +1 base)"},
    "fortify": {"type": "number", "description": "priority for FORTIFY (-2 ore, +1 shield)"},
    "raid": {"type": "number", "description": "per-ship priority for RAID"},
    "raid_threshold": {"type": "integer", "description": "min ships before raiding (1-20)"},
    "expand_late_bonus": {"type": "number", "description": "EXPAND grows toward endgame (0-5)"},
    "fortify_threat_scale": {"type": "number", "description": "FORTIFY grows with enemy fleet (0-2)"},
}


class ArenaToolServer:
    def __init__(self, league: League | None = None):
        self.league = league or League()
        self.call_log: List[Dict] = []

    # -- MCP surface -------------------------------------------------------- #
    def list_tools(self) -> List[Dict]:
        genome_schema = {"type": "object", "properties": _GENOME_PROPERTIES}
        return [
            {
                "name": "evaluate_strategy",
                "description": "Run a strategy genome through the calibrated Elo "
                               "league against all anchors. Returns elo, rank, "
                               "win-rate vs the top anchor, and whether it beats it.",
                "inputSchema": {"type": "object",
                                "properties": {"genome": genome_schema},
                                "required": ["genome"]},
            },
            {
                "name": "simulate_match",
                "description": "Play a genome head-to-head against one named anchor "
                               "across the map pool. Returns wins/draws/losses.",
                "inputSchema": {"type": "object",
                                "properties": {"genome": genome_schema,
                                               "opponent": {"type": "string"}},
                                "required": ["genome", "opponent"]},
            },
            {
                "name": "list_anchors",
                "description": "List the calibrated anchor ladder with Elo and notes.",
                "inputSchema": {"type": "object", "properties": {}},
            },
        ]

    def call_tool(self, name: str, arguments: Dict) -> Dict:
        self.call_log.append({"tool": name, "arguments": arguments})
        if name == "evaluate_strategy":
            return self._evaluate(arguments["genome"], arguments.get("name", "candidate"))
        if name == "simulate_match":
            return self._simulate(arguments["genome"], arguments["opponent"])
        if name == "list_anchors":
            return self._list_anchors()
        raise ValueError(f"unknown tool: {name}")

    # -- implementations ---------------------------------------------------- #
    def _evaluate(self, genome: Dict, name: str = "candidate") -> Dict:
        cand = Strategy(name=name, genome={k: genome[k] for k in genome if k in GENOME_KEYS})
        return self.league.evaluate_candidate(cand)

    def _simulate(self, genome: Dict, opponent: str) -> Dict:
        from arena.sim import play_match
        cand = Strategy(name="candidate", genome=genome)
        opp = next((a for a in self.league.anchors if a.name == opponent), None)
        if opp is None:
            return {"error": f"no such anchor '{opponent}'",
                    "anchors": [a.name for a in self.league.anchors]}
        wa, dr, wb = play_match(cand, opp, self.league.pool)
        return {"opponent": opponent, "wins": wa, "draws": dr, "losses": wb,
                "games": wa + dr + wb}

    def _list_anchors(self) -> Dict:
        res = self.league.run()
        return {"anchors": [
            {"name": a.name, "elo": round(res.elo[a.name], 1), "note": a.note}
            for a in sorted(self.league.anchors, key=lambda s: -res.elo[s.name])]}

    # -- convenience for the orchestrator ----------------------------------- #
    def anchor_summary(self) -> List[Dict]:
        return self._list_anchors()["anchors"]


In [ ]:
%%writefile agent/skills/strategy_heuristics.md
# Skill: Orbital Skirmish strategy heuristics

Distilled domain knowledge the Proposer loads as context before designing a genome.
A *skill* is just reusable, versioned expertise injected at the right moment — the
course Day-3 idea. These heuristics were learned by watching the calibrated league.

## How the game pays out
- **Bases are the only victory points.** You must EXPAND to win; pure economy draws 0–0.
- **Bases are fragile.** An undefended expander bleeds bases to any standing fleet
  (each net attacking ship destroys one base, every turn). Expansion without defence
  is the classic losing line (`greedy_expander`).
- **Defence is cheap and permanent.** A shield (cost 2) permanently blocks one
  attacking ship. Keeping `shields ≥ enemy fleet` makes you raid-proof.
- **Offence needs an economy.** Raiders with no mining (`all_in_raider`) run out of
  ore and stall; their fleet bounces off a fortified opponent.

## What tends to win
1. **Economy first, expand late.** Mine to a healthy ore bank, then convert to bases
   in the back half (`expand_late_bonus` 0.8–1.5).
2. **Reactive defence.** Don't over-fortify blindly; scale fortification to the
   opponent's fleet (`fortify_threat_scale` 0.5–1.0) so shields track the real threat.
3. **A credible deterrent, not all-in.** Keep `raid` priority moderate and
   `raid_threshold` low-ish (3–5) so you punish undefended expanders without
   starving your own economy.
4. **Balance beats extremes.** Every anchor that maxes one knob sits low on the
   ladder. The top anchor (`adaptive`) is balanced + situational.

## Falsified lines (don't re-propose — wasted budget)
- mine-only / build-only: never scores, bottom of the ladder.
- expand-max with zero fortify: punished hard by any fleet.
- raid-max with zero economy: stalls; loses to anyone who fortifies.

> Methodology note: these are *hypotheses ranked by the league*, not gospel. Always
> trust the Elo number over the heuristic — the calibrated eval is the ground truth.


In [ ]:
%%writefile agent/strategist.py
"""The orchestrator: a self-correcting Proposer → Eval → Critic loop.

This is the agent proper. Each round:

  1. **Proposer** (LLM or Mock) reads the skill, the anchor ladder, the best genome
     so far, recent attempts and the falsification log, and proposes a new genome.
  2. The orchestrator calls the **`evaluate_strategy` tool** (the MCP surface) to
     score it on the calibrated league — the objective reward signal.
  3. **Critic** (LLM or Mock) reads the evaluation and returns keep / mutate /
     discard, optionally suggesting knob changes — the self-correction step.
  4. Everything is written to **memory**; new dead ends become **falsifications**.

It is deliberately the same control loop as the Orbit Wars search that earned us
silver: a trustworthy eval in the inner loop, memory of what failed on the outside.
"""

from __future__ import annotations

import os
from dataclasses import dataclass, field
from typing import Dict, List, Optional

from arena.league import League
from arena.sim import Strategy

from .memory import Attempt, StrategyMemory
from .provider import ProposalContext, StrategyProvider, get_provider
from .tools import ArenaToolServer

_SKILL_PATH = os.path.join(os.path.dirname(__file__), "skills", "strategy_heuristics.md")


def _load_skill() -> str:
    try:
        with open(_SKILL_PATH, "r", encoding="utf-8") as f:
            return f.read()
    except OSError:
        return "(skill file unavailable)"


@dataclass
class RunReport:
    rounds: int
    best_name: str
    best_genome: Dict[str, float]
    best_elo: float
    top_anchor: str
    top_anchor_elo: float
    beat_ceiling: bool
    tool_calls: int
    provider: str
    trace: List[Dict] = field(default_factory=list)

    def headline(self) -> str:
        verb = "BEAT" if self.beat_ceiling else "did not beat"
        return (f"{self.provider} strategist {verb} the ceiling: best elo "
                f"{self.best_elo:.0f} vs {self.top_anchor} {self.top_anchor_elo:.0f} "
                f"after {self.rounds} rounds ({self.tool_calls} tool calls).")


class ArenaStrategist:
    def __init__(self, provider: Optional[StrategyProvider] = None,
                 league: Optional[League] = None,
                 memory_path: Optional[str] = None,
                 verbose: bool = True):
        self.league = league or League()
        self.tools = ArenaToolServer(self.league)
        self.provider = provider or get_provider()
        self.memory = StrategyMemory(memory_path)
        self.skill = _load_skill()
        self.verbose = verbose
        # Pin the ceiling once (anchors-only league) so the target is fixed.
        anchors = self.tools.anchor_summary()
        self.top = anchors[0]
        self.anchor_summary = anchors

    def _log(self, msg: str) -> None:
        if self.verbose:
            print(msg)

    def _context(self, round_index: int, budget: int) -> ProposalContext:
        return ProposalContext(
            top_anchor=self.top["name"],
            top_anchor_elo=self.top["elo"],
            anchor_summary=self.anchor_summary,
            skill_text=self.skill,
            best_genome=self.memory.best_genome(),
            best_elo=self.memory.best_elo(),
            history=self.memory.recent(6),
            falsifications=list(self.memory.falsifications),
            round_index=round_index,
            budget=budget,
        )

    def run(self, budget: int = 20) -> RunReport:
        self._log(f"Arena Strategist - provider={self.provider.name}, "
                  f"ceiling={self.top['name']} (elo {self.top['elo']:.0f}), budget={budget}\n")
        trace: List[Dict] = []

        for r in range(budget):
            ctx = self._context(r, budget)

            # 1. Proposer
            proposal = self.provider.propose(ctx)
            cand = Strategy(name=f"gen{r:02d}", genome=proposal.genome, note=proposal.rationale)

            # Skip exact repeats — don't spend eval budget twice on a known point.
            if self.memory.already_tried(cand.normalised()):
                self._log(f"[r{r:02d}] repeat genome skipped")
                continue

            # 2. Evaluate via the MCP tool surface (the reward signal)
            ev = self.tools.call_tool("evaluate_strategy",
                                      {"genome": cand.normalised(), "name": cand.name})

            # 3. Critic (self-correction)
            crit = self.provider.critique(cand, ev, ctx)

            # 4. Memory
            self.memory.record(Attempt(
                round_index=r, name=cand.name, genome=cand.normalised(),
                elo=ev["elo"], rank=ev["rank"], beats_top_anchor=ev["beats_top_anchor"],
                winrate_vs_top=ev["winrate_vs_top"], rationale=proposal.rationale,
                source=proposal.source, verdict=crit.verdict))
            if crit.verdict == "discard":
                self.memory.add_falsification(
                    f"{proposal.rationale[:70]} -> elo {ev['elo']:.0f} (weak); abandoned.")

            flag = "* NEW BEST" if self.memory.best.round_index == r else ""
            beat = "BEATS ceiling" if ev["beats_top_anchor"] else ""
            self._log(f"[r{r:02d}] {proposal.source:<6} elo {ev['elo']:>6.0f} "
                      f"rank {ev['rank']}/{ev['field_size']} "
                      f"wr_vs_{self.top['name']} {ev['winrate_vs_top']:>4.0%} "
                      f"-> {crit.verdict:<7} {beat} {flag}")
            trace.append({"round": r, "source": proposal.source, "elo": ev["elo"],
                          "rank": ev["rank"], "beats_top": ev["beats_top_anchor"],
                          "verdict": crit.verdict, "rationale": proposal.rationale})

        best = self.memory.best
        report = RunReport(
            rounds=len(trace),
            best_name=best.name, best_genome=best.genome, best_elo=best.elo,
            top_anchor=self.top["name"], top_anchor_elo=self.top["elo"],
            beat_ceiling=best.elo > self.top["elo"],
            tool_calls=len(self.tools.call_log), provider=self.provider.name, trace=trace)
        self._log("\n" + report.headline())
        return report


## 2. (Optional) Enable the real Gemini strategist
Skip this to run on the deterministic Mock. To use Gemini:
1. Add your AI Studio key as a Kaggle **Secret** named `GOOGLE_API_KEY`
   (Add-ons → Secrets), then enable it for this notebook; **or**
2. set the environment variable in-notebook (less secure):

```python
import os; os.environ["GOOGLE_API_KEY"] = "..."   # and: pip install google-generativeai
```
The model defaults to `gemini-2.0-flash`; override with `GEMINI_MODEL`.

## 3. Calibrate the league (the trustworthy evaluator)
Before judging any new strategy, confirm the league reproduces a sensible ordering
of the **anchors** — naive at the bottom, the sophisticated `adaptive` anchor on top.
This calibration step is the whole reason the eval can be trusted; it's the single
biggest lesson we carried over from Orbit Wars.

In [ ]:
from arena import League, Strategy
league = League()
print(league.run().table())
print("\nCeiling to beat:", league.top_anchor().name)

## 4. The agent climbs the ladder
Each round: **Proposer** suggests a genome → the **`evaluate_strategy` tool** scores
it on the league → the **Critic** says keep / mutate / discard → everything goes to
**memory**, and dead ends become **falsifications** the Proposer won't revisit.

In [ ]:
from agent import ArenaStrategist, get_provider

provider = get_provider()          # Gemini if a key is present, else Mock
agent = ArenaStrategist(provider=provider, league=league, memory_path="arena_memory.json")
report = agent.run(budget=24)

## 5. Verify the champion & visualise the climb
Drop the discovered champion back into a fresh league to confirm it really tops the
ladder, then plot Elo per round.

In [ ]:
import matplotlib.pyplot as plt

champ = Strategy(name="champion", genome=report.best_genome, note="discovered by the agent")
print(league.run(extra=[champ]).table())
print("\nChampion genome:", report.best_genome)
print(report.headline())

elos = [t["elo"] for t in report.trace]
rounds = [t["round"] for t in report.trace]
plt.figure(figsize=(9, 4))
plt.plot(rounds, elos, marker="o", label="proposed genome")
plt.axhline(report.top_anchor_elo, ls="--", color="crimson",
            label=f"ceiling ({report.top_anchor} {report.top_anchor_elo:.0f})")
best = report.top_anchor_elo
run_best = []
for e in elos:
    best = max(best if run_best else e, e) if not run_best else max(run_best[-1], e)
    run_best.append(best)
plt.plot(rounds, run_best, color="green", lw=2, label="best so far")
plt.xlabel("round"); plt.ylabel("Elo"); plt.title("Arena Strategist — climb vs the ceiling")
plt.legend(); plt.tight_layout(); plt.show()

print("\n" + agent.memory.summary())
for f in agent.memory.falsifications:
    print("  falsified:", f)

## 6. The MCP tool surface (inspect)
The agent only ever touches the simulator through these schema-typed tools — the
same shape as a real Model Context Protocol server (`tools/list` + `tools/call`).

In [ ]:
import json
from agent import ArenaToolServer
ts = ArenaToolServer(league)
print(json.dumps(ts.list_tools(), indent=2)[:1400], "...")
print("\nExample call -> simulate_match vs the top anchor:")
print(ts.call_tool("simulate_match", {"genome": report.best_genome, "opponent": report.top_anchor}))

## 7. How it works — architecture & design notes

```
        +------------------ ArenaStrategist (orchestrator loop) ------------------+
        |                                                                         |
  skill + memory ---> [ Proposer ] --genome--> [ evaluate_strategy TOOL ] --eval--+
   (context)              (Gemini)                  (calibrated league)            |
        ^                                                                         v
        |                                                       [ Critic ] keep/mutate/discard
        +----------------------- memory (attempts + falsifications) <-------------+
```

**Why a calibrated league is the heart of it.** An LLM will happily invent
plausible-sounding strategies; most are wrong. The only way to know is an evaluator
you trust. We first calibrate the league against hand-built anchors of known
relative strength — if it can't reproduce *their* ordering, it can't be trusted on
novel strategies. Only then do we let the agent optimise against it. This is exactly
how we avoided fooling ourselves in Orbit Wars, where a *miscalibrated* local eval
had previously sent the search chasing mirages.

**Why strategies are genomes, not code.** Representing a strategy as 8 numbers (not
free-form code) makes proposals safe to execute, trivial to store/diff/mutate, and
easy for an LLM to emit as strict JSON — no sandbox required. (The course starter
kernel *does* show the code-generation-plus-sandbox pattern; we note it as the
natural Day-4 security extension if strategies were arbitrary code.)

**Memory & falsification.** Every attempt is persisted; genuinely bad lines become
short "falsification" notes fed back into the Proposer's context. Writing down what
*doesn't* work is the cheapest way to stop an agent (or a human) re-exploring dead
ends — the highest-leverage habit from our competition work.

**Graceful degradation (production thinking).** Missing key → Mock. Unparseable LLM
output → coerced/fallback genome. The loop never stalls; the notebook always runs.

### Connection to Orbit Wars
Orbit Wars is our $50k Kaggle simulation result (silver, top 5% of 4,689). It is a
pure reinforcement-learning / search agent — no LLM. This capstone is **not** that
agent; it is the *methodology* from it, re-cast as a vibe-coded LLM agent on a
self-contained arena: calibrate a trustworthy league, then search against it with a
memory of dead ends. (Orbit Wars' live submission is locked and untouched here.)

### Limitations & next steps
- The arena is deliberately compact; the policy is argmax over weighted actions, so
  the genome→Elo surface is piecewise-constant (many genomes tie). A richer policy
  (state-conditioned, or LLM-authored code in a sandbox) would deepen the search.
- The league is single-threaded; Day-5 production would parallelise matches and add
  tracing/metrics export.
- Self-play co-evolution (promote champions into the anchor set, then re-climb) is
  the natural extension — exactly the AlphaZero-lite direction we explored in Orbit Wars.

*Built for the Google × Kaggle 5-Day AI Agents Intensive (Vibe Coding) capstone.*